In [69]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from constants import tratados_dir

In [70]:
df_test = pd.read_csv(f'{tratados_dir}/vehiculos_test.csv')
df_train = pd.read_csv(f'{tratados_dir}/vehiculos_train.csv')

### Conversion de la variable categorica **marca** usando **target encoding**

In [71]:
%pip install category_encoders

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [72]:
from category_encoders import TargetEncoder

Instanciamos y entrenamos TargetEncoder con los datos de entrenamiento.

In [73]:
encoder = TargetEncoder(cols=['marca'])

Procedemos con la conversion de los datos.

In [74]:
df_train['marca_te'] = encoder.fit_transform(df_train['marca'], df_train['fraude'])
df_test['marca_te'] = encoder.transform(df_test['marca'])

### Conversion de la variable categorica **transmision**

En este caso hemos identifica una relacion entre los tipos de transmision. Al menos entre **manual-semiautomatica-automático**. Con **híbrido** no tenemos claro si existe una relacion con las categorias anteriores por lo que lo trataremos como un caso aislado en la convertibilidad.

Este es el mapa de los tipos de transmision que planteamos donde **semi-automatic** es un punto medio entre **manual** y **automatico**.

In [75]:
transmission_map = {
    'Manual': 0,
    'Semi-Automatic': 0.5,
    'Automatic': 1,
    'Hybrid': 2
}

Procedemos a la conversion de los valores de la variable **transmision**

In [76]:
df_train['transmision_encoded'] = df_train['transmision'].map(transmission_map)
df_test['transmision_encoded'] = df_test['transmision'].map(transmission_map)

### Conversion de la variable  categorica **tipo_combustible**

In [77]:
print(df_train['tipo_combustible'].unique())

['Petrol' 'Diesel' 'Petrol Hybrid' 'Hybrid  Petrol/Electric'
 'Hybrid  Petrol/Electric Plug-in' 'Electric' 'Hybrid  Diesel/Electric'
 'Petrol Plug-in Hybrid' 'Diesel Hybrid' 'Hydrogen'
 'Hybrid  Diesel/Electric Plug-in' 'Bi Fuel' 'Petrol Ethanol'
 'Diesel Plug-in Hybrid']


Vemos que tenemos  vehículos que son híbiridos en cuanto al tipo de conbustible. En este caso aplicaremos un **one-hot-encoding**, pero no directamente a las valorres unicos presentados en el campo anterior, sino que los vamos a descomponer y ponerlos en una representacion binaria como se verá a continuación:

Los de combustion única deberían tener solo un 1 entre sus columnas, mientras que los dos 1.

In [78]:
def fuel_features(fuel):
    return pd.Series({
        'fuel_petrol': int('Petrol' in fuel),
        'fuel_diesel': int('Diesel' in fuel),
        'fuel_electric': int('Electric' in fuel),
        'fuel_hybrid': int('Hybrid' in fuel),
        'fuel_plugin': int('Plug-in' in fuel),
        'fuel_other': int(all(x not in fuel for x in ['Petrol', 'Diesel', 'Electric', 'Hybrid', 'Plug-in']))
    })

In [79]:
df_train = pd.concat([df_train, df_train['tipo_combustible'].apply(fuel_features)], axis=1)
df_test = pd.concat([df_test, df_test['tipo_combustible'].apply(fuel_features)], axis=1)

### Conversion de la variable color

In [80]:
print(df_train['color'].unique())

['Black' 'Grey' 'White' 'Blue' 'Yellow' 'Red' 'Orange' 'Silver' 'Gelb'
 'Beige' 'Green' 'Multicolour' 'Brown' 'Gold' 'Purple' 'Bronze' 'Pink'
 'Turquoise' 'Maroon' 'Burgundy' 'Magenta' 'Navy' 'Indigo']


Sabemos que los colores tienen relacion entre ellos. En este caso, utilizaremos la representacion vectorial RBG para representar la relacion entre ellos.

In [81]:
color_rgb_map = {
    'Black': (0, 0, 0),
    'Grey': (128, 128, 128),
    'White': (255, 255, 255),
    'Blue': (0, 0, 255),
    'Yellow': (255, 255, 0),
    'Red': (255, 0, 0),
    'Orange': (255, 165, 0),
    'Silver': (192, 192, 192),
    'Gelb': (255, 255, 0),  # 'Gelb' is German for yellow
    'Beige': (245, 245, 220),
    'Green': (0, 128, 0),
    'Multicolour': (128, 128, 128),  # or average of all colors
    'Brown': (139, 69, 19),
    'Gold': (255, 215, 0),
    'Purple': (128, 0, 128),
    'Bronze': (205, 127, 50),
    'Pink': (255, 192, 203),
    'Turquoise': (64, 224, 208),
    'Maroon': (128, 0, 0),
    'Burgundy': (128, 0, 32),
    'Magenta': (255, 0, 255),
    'Navy': (0, 0, 128),
    'Indigo': (75, 0, 130)
}

In [82]:
def color_to_rgb(color):
    return color_rgb_map.get(color, (128, 128, 128))  # Default: grey

df_train[['color_r', 'color_g', 'color_b']] = df_train['color'].apply(lambda x: pd.Series(color_to_rgb(x)))
df_test[['color_r', 'color_g', 'color_b']] = df_test['color'].apply(lambda x: pd.Series(color_to_rgb(x)))

### Guardado de los datos convertidos

Procedemos a guardar estos datos

Limpiamos el dataset de las antiguas columnas.

In [83]:
df_train = df_train.drop(['marca','transmision','tipo_combustible','color'], axis=1 )
df_test = df_test.drop(['marca','transmision','tipo_combustible','color'], axis=1)

In [84]:
import os
from constants import converted_dir

In [85]:
os.makedirs(converted_dir, exist_ok=True)

In [86]:
df_train.to_csv(f"{converted_dir}/vehiculos_train.csv", index=False)
df_test.to_csv(f"{converted_dir}/vehiculos_test.csv", index=False)